### Context and State

In [1]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

Globals

In [2]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Creating State

In [3]:
class CustomState(AgentState):
    fav_color: str

Tool for State

In [4]:
@tool
def update_color(fav_color: str, runtime: ToolRuntime) -> Command:
    """Update Color

    Args:
        fav_color (str): favourite color
        runtime (ToolRuntime): agent runtime

    Returns:
        Command: command to be returned
    """
    
    
    return Command(
        update={
            "fav_color": fav_color,
            "messages": [ToolMessage("Updated color", tool_call_id=runtime.tool_call_id)]
        }
    )

Agent

In [5]:
agent = create_agent(
    model=model, 
    tools=[update_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Updating State Through The Prompt

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

for m in response['messages']:
    m.pretty_print()

================================ Human Message =================================

My favourite colour is green
================================== Ai Message ==================================



I'll update your favourite colour to green right away!
Tool Calls:
  update_color (toolu_bdrk_01BrMZKK6NFAVDcw2U94Mh1C)
 Call ID: toolu_bdrk_01BrMZKK6NFAVDcw2U94Mh1C
  Args:
    fav_color: green
================================= Tool Message =================================
Name: update_color

Updated color
================================== Ai Message ==================================

Your favourite colour has been updated to **green**! 🟢 Is there anything else you'd like to update?
{'fav_color': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='b15c60bb-12dd-4d51-9384-2ad0eedfb5e9'),
              AIMessage(content="\n\nI'll update your favourite colour to green right away!", additional_kwargs={'usage': {'prompt_toke

Changing State at Agent Invocation

In [ ]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "fav_color": "red"
    },
    {"configurable": {"thread_id": "10"}}
)

for m in response['messages']:
    m.pretty_print()

NameError: name 'Globals' is not defined

Reading State

In [ ]:
@tool
def read_color(runtime: ToolRuntime) -> str:
    """read the favourite color of the user from the state"""
    try:
        return runtime.state["fav_color"]
    except KeyError:
        return "no favourite colour found in state"

Agent

In [ ]:
agent = create_agent(
    model=model, 
    tools=[update_color, read_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Updating State and Reading from State

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

for m in response['messages']:
    m.pretty_print()

{'fav_color': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='040f048b-64b1-41fe-af95-222449a5f5e3'),
              AIMessage(content="\n\nI'll update your favourite colour to green right away!", additional_kwargs={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde1e-b69e-7eb2-a5a3-2fc36e87da93-0', tool_calls=[{'name': 'update_color', 'args': {'fav_color': 'green'}, 'id': 'toolu_bdrk_01EDykF9n3hRy2Vjp2QCv5kg', 'type': 'tool_c

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

for m in response['messages']:
    m.pretty_print()

SyntaxError: '(' was never closed (912838870.py, line 1)